# Drug-Drug Interaction — Explainability
This notebook demonstrates GNNExplainer (graph-level) and SHAP (molecular feature-level) explanations.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

sys.path.append("..")

from src.model import MLPBaseline, GCNModel
from src.dataset import load_twosides, build_drug_vocab

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load Models and Graph

In [ ]:
data = torch.load("../data/processed/ddi_graph.pt", weights_only=False)
x = data.x.to(device)
edge_index = data.edge_index.to(device)

mlp = MLPBaseline(input_dim=2048).to(device)
mlp.load_state_dict(
    torch.load("../models/best_mlp.pt", map_location=device, weights_only=False)
)
mlp.eval()

gcn = GCNModel(input_dim=2048).to(device)
gcn.load_state_dict(
    torch.load("../models/best_gcn.pt", map_location=device, weights_only=False)
)
gcn.eval()

pairs = load_twosides("../data/raw/TWOSIDES.csv")
drug2idx = build_drug_vocab(pairs)
idx2drug = {v: k for k, v in drug2idx.items()}

drugs = list(drug2idx.keys())
drug1, drug2 = drugs[0], drugs[1]
idx1, idx2 = drug2idx[drug1], drug2idx[drug2]

print(f"Explaining: {drug1} + {drug2}")
print(f"Indices: {idx1}, {idx2}")

## 2. Prediction Scores

In [ ]:
def predict(model, x, edge_index, i, j):
    target = torch.tensor([[i], [j]], dtype=torch.long).to(device)
    with torch.no_grad():
        return torch.sigmoid(model(x, edge_index, target)).item()


mlp_score = predict(mlp, x, edge_index, idx1, idx2)
gcn_score = predict(gcn, x, edge_index, idx1, idx2)

print(f"MLP score: {mlp_score * 100:.1f}%")
print(f"GCN score: {gcn_score * 100:.1f}%")
print(f"\nDifference: {abs(mlp_score - gcn_score) * 100:.1f}%")
print("MLP uses molecular features only; GCN incorporates graph neighborhood")

## 3. SHAP Explanation on MLP
SHAP identifies which of the 2048 Morgan fingerprint bits most influenced the MLP prediction.

In [ ]:
import shap

x_cpu = x.cpu()
np.random.seed(42)
bg1 = np.random.randint(0, x_cpu.shape[0], 50)
bg2 = np.random.randint(0, x_cpu.shape[0], 50)
background = np.hstack([x_cpu[bg1].numpy(), x_cpu[bg2].numpy()])
pair_feat = np.hstack([x_cpu[idx1].numpy(), x_cpu[idx2].numpy()]).reshape(1, -1)


def model_predict(pair_features):
    tensor = torch.tensor(pair_features, dtype=torch.float).to(device)
    with torch.no_grad():
        h = torch.relu(mlp.fc1(tensor))
        h = mlp.dropout(h)
        h = torch.relu(mlp.fc2(h))
        h = mlp.dropout(h)
        out = torch.sigmoid(mlp.fc3(h))
    return out.cpu().numpy()


print("Running SHAP (~30 seconds)...")
explainer = shap.KernelExplainer(model_predict, background)
shap_values = explainer.shap_values(pair_feat, nsamples=100)
print("Done!")

## 4. SHAP Feature Importance Plot

In [ ]:
shap_drug1 = shap_values[0][:2048]
shap_drug2 = shap_values[0][2048:]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, shap_vals, label in [
    (axes[0], shap_drug1, f"{drug1}\n(Drug 1)"),
    (axes[1], shap_drug2, f"{drug2}\n(Drug 2)"),
]:
    top_bits = np.argsort(np.abs(shap_vals))[::-1][:15]
    top_vals = shap_vals[top_bits]
    colors = ["#e74c3c" if v > 0 else "#3498db" for v in top_vals]
    ax.barh(range(15), top_vals[::-1], color=colors[::-1])
    ax.set_yticks(range(15))
    ax.set_yticklabels([f"Bit {b}" for b in top_bits[::-1]], fontsize=9)
    ax.set_title(f"Top 15 SHAP Features\n{label}")
    ax.set_xlabel("SHAP Value")
    ax.axvline(0, color="black", linewidth=0.8)

plt.suptitle(
    "SHAP Feature Importance — Molecular Fingerprint Bits",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig("../data/processed/shap_explanation.png", dpi=150)
plt.show()
print("Red = increases interaction probability | Blue = decreases")

## 5. GNNExplainer on GCN
GNNExplainer identifies which neighboring drugs and graph edges are most influential.

In [ ]:
from torch_geometric.explain import Explainer, GNNExplainer


class GCNWrapper(torch.nn.Module):
    def __init__(self, model, target_edge):
        super().__init__()
        self.model = model
        self.target_edge = target_edge

    def forward(self, x, edge_index):
        return self.model(x, edge_index, self.target_edge)


target_edge = torch.tensor([[idx1], [idx2]], dtype=torch.long).to(device)
wrapped = GCNWrapper(gcn, target_edge)

explainer = Explainer(
    model=wrapped,
    algorithm=GNNExplainer(epochs=100),
    explanation_type="model",
    node_mask_type="attributes",
    edge_mask_type="object",
    model_config=dict(
        mode="binary_classification", task_level="graph", return_type="raw"
    ),
)

print("Running GNNExplainer...")
explanation = explainer(x=x, edge_index=edge_index)
print("Done!")

## 6. Top Influential Neighbor Drugs

In [ ]:
if explanation.node_mask is not None:
    node_importance = explanation.node_mask.sum(dim=1).cpu().numpy()
    top_nodes = np.argsort(node_importance)[::-1][:10]

    df_nodes = pd.DataFrame(
        {
            "Drug ID": [idx2drug[n] for n in top_nodes],
            "Node Index": top_nodes,
            "Importance Score": node_importance[top_nodes].round(4),
            "Role": [
                "Target Drug 1"
                if n == idx1
                else "Target Drug 2"
                if n == idx2
                else "Neighbor"
                for n in top_nodes
            ],
        }
    )

    print("Top 10 most influential drug nodes:")
    print(df_nodes.to_string(index=False))

    plt.figure(figsize=(10, 5))
    colors = ["#e74c3c" if r != "Neighbor" else "#3498db" for r in df_nodes["Role"]]
    plt.barh(df_nodes["Drug ID"], df_nodes["Importance Score"], color=colors)
    plt.title("GNNExplainer — Most Influential Drug Nodes")
    plt.xlabel("Importance Score")
    plt.tight_layout()
    plt.savefig("../data/processed/gnn_explanation.png", dpi=150)
    plt.show()

## 7. Summary

In [ ]:
print("=" * 50)
print("EXPLAINABILITY SUMMARY")
print("=" * 50)
print(f"Drug pair: {drug1} + {drug2}")
print(f"MLP prediction: {mlp_score * 100:.1f}% (molecular features only)")
print(f"GCN prediction: {gcn_score * 100:.1f}% (graph neighborhood)")
print()
print("SHAP  → explains MLP: which fingerprint bits matter most")
print("GNNEx → explains GCN: which neighbor drugs matter most")
print()
print("Together: complete molecular + network level explanation")